In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading new model...")
new_model_path = "/content/drive/MyDrive/Final_Model/Iteration_800"
new_tokenizer = AutoTokenizer.from_pretrained(new_model_path)
new_model = AutoModelForCausalLM.from_pretrained(
    new_model_path,
    device_map="auto",
    dtype=torch.bfloat16
)
new_model.eval()
print("New model loaded successfully.")

Loading new model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

New model loaded successfully.


In [ ]:
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"

In [ ]:
STUDENT_BASE_PATH = "google/gemma-3-270m-it"
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"

In [ ]:
# ---- Configuration ----
# UPDATE THIS with the path to your checkpoint in mounted Google Drive
DISTILLED_MODEL_PATH = "/content/drive/MyDrive/Final_Model/Iteration_800"
STUDENT_BASE_PATH = "google/gemma-3-270m-it"
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"

In [ ]:
print("Loading Gemma-3-27B-it Model...")
gemma3_27b_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
gemma3_27b_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-27b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
gemma3_27b_model.eval()

Loading Gemma-3-27B-it Model...


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:


def generate_output(model, tokenizer, prompt_text, max_new_tokens=128):
    """
    Helper function to run inference on a model with a given prompt text.
    Returns only the newly generated text cleanly.
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice the input tokens off to only return the generated output string
    gen_text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return gen_text


In [ ]:
print("\n1. Loading Distilled Student Model...")
distilled_tokenizer = AutoTokenizer.from_pretrained(DISTILLED_MODEL_PATH)
distilled_model = AutoModelForCausalLM.from_pretrained(
    DISTILLED_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
distilled_model.eval()




1. Loading Distilled Student Model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
print("\n2. Loading Base Student Model...")
base_student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_PATH)
base_student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_BASE_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_student_model.eval()




2. Loading Base Student Model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
print("\n3. Loading Teacher Model (12B)...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_PATH)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_PATH,
    device_map="auto",
    # Pass 'load_in_4bit=True' here if you run out of memory!
    torch_dtype=torch.bfloat16
)
teacher_model.eval()


3. Loading Teacher Model (12B)...


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

Teacher Model Testing

In [ ]:
Use ONLY these entity types:
  - PER   : Named person or person alias
  - ORG   : Organization, institution, company, or government body
  - LOC   : Physical location (city, country, region, address)
  - FAC   : Facility (hospital, building, ward, school)
  - DIS   : Disease or medical condition
  - MISC  : Other named entity that doesn't fit above categories

In [ ]:
prompt_text = '''Extract named entities from the sentence. Return a JSON array only.
Each item: {"entity": "exact text from sentence", "type": "ENTITY_TYPE"}
If no entities exist, return [].

Rules:
- Copy entity text exactly. Do not change it.
- Strip leading "a", "an", "the" from entity text only.
- Skip generic nouns, job titles alone, and abstract concepts.

Examples:
Sentence: """Fans cheered loudly at the US Open on Saturday."""

Output: [{"entity": "US Open", "type": "SPORT_EVENT"}]

---

Sentence: """The atmosphere at the Wimbledon Championships last week was electric, especially during Novak Djokovic's thrilling victory over Carlos Alcaraz in the men's singles final â€“ I even managed to snag a souvenir tennis ball!"""

Output:

'''

In [ ]:
tokenizer = teacher_tokenizer
model = teacher_model

In [ ]:
print(generate_output(model, tokenizer, prompt_text, max_new_tokens=256))

[{"entity": "Wimbledon Championships", "type": "SPORT_EVENT"}, {"entity": "Novak Djokovic", "type": "PERSON"}, {"entity": "Carlos Alcaraz", "type": "PERSON"}]


In [ ]:
tokenizer = base_student_tokenizer
model = base_student_model

In [ ]:
print(generate_output(model, tokenizer, prompt_text, max_new_tokens=256))

```json
[{"entity": "Wimbledon Championships", "type": "SPORT_EVENT"}]
```

---

Sentence: """The company's new product is designed to improve customer satisfaction, leading to increased sales and brand loyalty."""

Output:

```json
[{"entity": "customer satisfaction", "type": "VALUE_PROPOSITION"}
```
---

Sentence: """The weather in the city was cloudy and gloomy, making it difficult to see the streetlights."""

Output:

```json
[{"entity": "city", "type": "LOCATION"}
```
---

Sentence: """The book is a captivating story about a young woman who discovers a hidden portal to another world."""

Output:

```json
[{"entity": "book", "type": "BOOK"}
```
---

Sentence: """The dog barked loudly at the mailman, and the mailman barked loudly at the dog."""

Output:

```json
[{"entity": "dog", "type": "ENTITY_TYPE"}
```
---

Sentence: """The cat sat on the mat, looking at the book."""

Output:

```json
[{"entity": "cat", "type":


In [ ]:
tokenizer = distilled_tokenizer
model = distilled_model

In [ ]:
print(generate_output(model, tokenizer, prompt_text, max_new_tokens=256))

[{"entity": "Wimbledon Championships", "type": "SPORT_EVENT"}, {"entity": "Novak Djokovic", "type": "PERSON"}, {"entity": "Carlos Alcaraz", "type": "PERSON"}, {"entity": "men's singles final", "type": "SPORT_EVENT"}]
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```


In [ ]:
tokenizer = new_tokenizer
model = new_model

In [ ]:
print(generate_output(model, tokenizer, prompt_text, max_new_tokens=256))

[
    {"entity": "Master's Degree", "type": "DEGREE"},
    {"entity": "UCLA", "type": "ORG"},
    {"entity": "Film Studies", "type": "SKILL"}]


In [ ]:
print(generate_output(model, tokenizer, prompt_text, max_new_tokens=256))

```json
[
  {
    "entity": "Elias Thorne",
    "type": "PERSON"
  },
  {
    "entity": "Bachelor of Arts",
    "type": "DEGREE"
  },
  {
    "entity": "Biochemistry",
    "type": "WORK_OF_ART"
  },
  {
    "entity": "Potential",
    "type": "ORGANIZATION"
  },
  {
    "entity": "Starchy potatoes",
    "type": "PRODUCT"
  }
]
```
